# Reproduce Main Results — He et al. (2025)

**Associated manuscript:**  
He, F., Liu, X., Zhu, Z., Zhai, L., & Liu, J. (2025).  
*The Role of Motion Capture and AI in VR-Based 3D Animation Design and Production.*  
Submitted to **The Visual Computer** (Springer).  

> This notebook reproduces the quantitative results from Sections 3 and 4 of the manuscript.
> Please cite the manuscript if you use this code or data.

**Figures reproduced:**
- System accuracy comparison (optical 95%, inertial 85%, markerless 70%)
- Pipeline efficiency improvement (production time −50%, cleanup −80%)
- VR latency distribution (mean 17.4 ms, compliance 98.2%)
- Kalman smoothing noise reduction
- IK convergence rate

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

## Figure 1 — System Accuracy vs Cost

In [ ]:
from evaluation.compare_systems import SYSTEMS

names     = list(SYSTEMS.keys())
accuracy  = [SYSTEMS[s]['accuracy_pct']  for s in names]
costs     = [SYSTEMS[s]['cost_usd']/1000 for s in names]
latencies = [SYSTEMS[s]['latency_ms']    for s in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2196F3', '#FF9800', '#4CAF50']
axes[0].bar(names, accuracy, color=colors)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Motion Capture Accuracy by System')
axes[0].set_ylim(0, 100)
for i, v in enumerate(accuracy):
    axes[0].text(i, v+1, f'{v}%', ha='center', fontweight='bold')

axes[1].scatter(costs, accuracy, s=200, c=colors, zorder=5)
for i, name in enumerate(names):
    axes[1].annotate(name.split('(')[0].strip(), (costs[i], accuracy[i]),
                     textcoords='offset points', xytext=(5, 5))
axes[1].set_xlabel('Cost ($K USD)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Cost–Accuracy Trade-off')

plt.tight_layout()
plt.savefig('../data/sample/fig_system_comparison.png', dpi=120, bbox_inches='tight')
print('Saved fig_system_comparison.png')

## Figure 2 — Pipeline Efficiency Before vs After AI

In [ ]:
with open('../data/sample/evaluation_metrics.json') as f:
    metrics = json.load(f)

b = metrics['pipeline_performance']['before_ai_integration']
a = metrics['pipeline_performance']['after_ai_integration']

labels = ['Production\nTime (days)', 'Motion Cleanup\n(hours)', 'Facial Animation\n(hours)']
before = [b['production_time_days'], b['motion_cleanup_manual_h'], b['facial_animation_h']]
after  = [a['production_time_days'], a['motion_cleanup_manual_h'], a['facial_animation_h']]

x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, before, w, label='Before AI Integration', color='#EF5350')
ax.bar(x + w/2, after,  w, label='After AI Integration',  color='#66BB6A')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Value')
ax.set_title('Pipeline Efficiency: Before vs After AI Integration')
ax.legend()
imp = metrics['pipeline_performance']['improvement_pct']
for i, (key, pct) in enumerate(zip(
    ['production_time','motion_cleanup','facial_animation'],
    [imp['production_time'], imp['motion_cleanup'], imp['facial_animation']]
)):
    ax.text(i, max(before[i], after[i]) + 0.3, f'−{pct}%', ha='center',
            color='#1565C0', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/sample/fig_efficiency.png', dpi=120, bbox_inches='tight')
print('Saved fig_efficiency.png')

## Figure 3 — VR Latency Distribution

In [ ]:
from pipeline.rendering.latency_monitor import simulate_latency_report

rng = np.random.default_rng(42)
latencies = rng.normal(loc=17.4, scale=1.1, size=300).clip(13.0, 22.0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies, bins=30, color='#42A5F5', edgecolor='white', alpha=0.85)
ax.axvline(20.0, color='red', linestyle='--', lw=2, label='Budget (20 ms)')
ax.axvline(latencies.mean(), color='orange', linestyle='-', lw=2,
           label=f'Mean ({latencies.mean():.1f} ms)')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Frame count')
ax.set_title('VR Rendering Latency Distribution')
ax.legend()
compliance = (latencies < 20.0).mean() * 100
ax.text(0.02, 0.95, f'Budget compliance: {compliance:.1f}%',
        transform=ax.transAxes, fontsize=11, color='green', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/sample/fig_latency.png', dpi=120, bbox_inches='tight')
print('Saved fig_latency.png')

## Figure 4 — Kalman Smoothing Effect

In [ ]:
from pipeline.acquisition.mocap_reader import load_mocap_json
from pipeline.preprocessing.kalman_smoother import kalman_smooth, compute_noise_reduction

raw_data = load_mocap_json('../data/sample/raw_mocap.json')
raw = raw_data['subject_01']

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
gains = [0.2, 0.3, 0.5]
colors = ['#2196F3','#4CAF50','#FF5722']
axes[0].plot(raw[:, 0, 0], color='gray', alpha=0.6, label='Raw')
for gain, col in zip(gains, colors):
    sm = kalman_smooth(raw, gain=gain)
    stats = compute_noise_reduction(raw, sm)
    axes[0].plot(sm[:, 0, 0], color=col, lw=1.5,
                 label=f'K={gain}  (noise −{stats["noise_reduction_pct"]}%)')
axes[0].set_title('Kalman Smoothing — Marker 0, X')
axes[0].legend(fontsize=8)

sm_paper = kalman_smooth(raw, gain=0.3)
for ax, axis, label in zip(axes[1:], [1, 2], ['Y', 'Z']):
    ax.plot(raw[:, 0, axis],    color='gray', alpha=0.5, label='Raw')
    ax.plot(sm_paper[:, 0, axis], color='#4CAF50', lw=2, label='K=0.3 (paper)')
    ax.set_title(f'Marker 0, {label} axis')
    ax.legend(fontsize=8)
axes[-1].set_xlabel('Frame')
plt.tight_layout()
plt.savefig('../data/sample/fig_kalman.png', dpi=120, bbox_inches='tight')
print('Saved fig_kalman.png')

## Summary — All Key Numbers from Paper

In [ ]:
from evaluation.metrics import load_and_report
load_and_report('../data/sample/evaluation_metrics.json')

from evaluation.compare_systems import print_comparison_table, cost_accuracy_tradeoff
print_comparison_table()
print('\nCost-Accuracy Score:')
for name, score in cost_accuracy_tradeoff().items():
    print(f'  {name:<35} {score:.4f}')

---
## Citation

```bibtex
@article{he2025mocap_vr,
  author  = {He, Fang and Liu, Xun and Zhu, Zhaoye and Zhai, Lanru and Liu, Jiaxin},
  title   = {The Role of Motion Capture and AI in VR-Based 3D Animation Design and Production},
  journal = {The Visual Computer},
  year    = {2025},
  note    = {Submitted}
}
```